# **Исследование доставки онлайн-заказов в Бразилии**

## **Цели исследовательской работы:**
- **Изучение успешных заказов.** Изучить, сколько заказов доставляется покупателю;
- **Изучение популярных маршрутов заказов.** Изучить заказы и доставки между регионами;
- **Изучение заказов, которые были доставлены с опозданием.** Изучить причины задержек в доставке и определить их причины.

In [1]:
# Импорт библиотек

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [2]:
# Форматируем float-данные для всей тетради

pd.set_option('display.float_format', '{:.2f}'.format)

In [3]:
customers = pd.read_csv("data/olist_customers_dataset.csv")
geo = pd.read_csv("data/olist_geolocation_dataset.csv")
order_items = pd.read_csv("data/olist_order_items_dataset.csv", parse_dates=["shipping_limit_date"])
order_payments = pd.read_csv("data/olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(
    "data/olist_order_reviews_dataset.csv", parse_dates=["review_creation_date", "review_answer_timestamp"]
)
orders = pd.read_csv(
    "data/olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
)
products = pd.read_csv("data/olist_products_dataset.csv")
sellers = pd.read_csv("data/olist_sellers_dataset.csv")
product_category_name = pd.read_csv("data/product_category_name_translation.csv")

## **1. Знакомство с датасетом и исследование данных**

### 1.1 Изменение типов данных с оптимизацией памяти

In [4]:
# Словарь с integer
cols_to_integer = {
    "customers": ["customer_zip_code_prefix"],
    "geo": ["geolocation_zip_code_prefix"],
    "order_items": ["order_item_id"],
    "order_payments": ["payment_sequential", "payment_installments"],
    "order_reviews": ["review_score"],
    "products": [
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
    ],
    "sellers": ["seller_zip_code_prefix"],
}

# Словарь с float
cols_to_float = {
    "geo": ["geolocation_lat", "geolocation_lng"],
    "order_items": ["price", "freight_value"],
    "order_payments": ["payment_value"],
}

# Словарь с датафреймами
dfs = {
    "customers": customers,
    "geo": geo,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "products": products,
    "sellers": sellers,
}

In [5]:
# Преобразуем данные

for df, columns in cols_to_integer.items():
    if df in dfs:
        df_copy = dfs[df]
        for col in columns:
            df_copy[col] = pd.to_numeric(df_copy[col], errors="coerce", downcast="integer")

for df, columns in cols_to_float.items():
    if df in dfs:
        df_copy = dfs[df]
        for col in columns:
            df_copy[col] = pd.to_numeric(df_copy[col], errors="coerce", downcast="integer")

In [6]:
display("customers", customers.head(), customers.info())
display("geo", geo.head(), geo.info())
display("order_items", order_items.head(), order_items.info())
display("order_payments", order_payments.head(), order_payments.info())
display("order_reviews", order_reviews.head(), order_reviews.info())
display("orders", orders.head(), orders.info())
display("products", products.head(), products.info())
display("sellers", sellers.head(), sellers.info())
display("product_category_name", product_category_name.head(), product_category_name.info())

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int32
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int32(1), str(4)
memory usage: 3.4 MB


'customers'

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


None

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int32  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int32(1), str(2)
memory usage: 34.3 MB


...

None

<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          71 non-null     str  
 1   product_category_name_english  71 non-null     str  
dtypes: str(2)
memory usage: 1.2 KB


'product_category_name'

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


None

### 1.2 Пропуски

#### Датафрейм `orders`

In [7]:
df_order_stats = orders[orders['order_delivered_customer_date'].isna()].groupby('order_status')['order_id'].nunique().to_frame(name='delivered')
df_order_stats['approved'] = orders[orders['order_approved_at'].isna()].groupby('order_status')['order_id'].nunique()
df_order_stats['carrier'] = orders[orders['order_delivered_carrier_date'].isna()].groupby('order_status')['order_id'].nunique()

df_order_stats = df_order_stats.fillna(0).astype(int)
df_order_stats = df_order_stats[['approved', 'carrier', 'delivered']]
df_order_stats

,approved,carrier,delivered
order_status,,,
approved,0,2,2
canceled,141,550,619
created,5,5,5
delivered,14,2,8
invoiced,0,314,314
processing,0,301,301
shipped,0,0,1107
unavailable,0,609,609


В датафрейме `orders` есть 3 колонки с пропусками. Пропуски почти во всех данных выглядят правдоподобно, кроме:

- `order_approved_at` (дата и время подтверждения заказа):
    - 14 в доставке. *Аномалия. Заказ уже доставляется, а подтверждения заказа не было. Скорее всего данные были повреждены или не записались в системе*
- `order_delivered_carrier_date` (дата и время передачи заказа в доставку):
    - 2 заказа в доставке; *Аномалия. Заказ доставляется, а время передачи в доставку не зафиксировано*
- `order_delivered_customer_date` (дата и время доставки покупателю): 
    - *Аномалий нет*

In [8]:
order_anomaly = orders[
    (orders["order_approved_at"].isna()) & (orders["order_status"] == "delivered")
    | (orders["order_delivered_carrier_date"].isna()) & (orders["order_status"] == "delivered")
]
order_anomaly

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaT,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaT,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaT,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaT,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaT,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20
26800,c1d4211b3dae76144deccd6c74144a88,684cb238dc5b5d6366244e0e0776b450,delivered,2017-01-19 12:48:08,NaT,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01
38290,d69e5d356402adc8cf17e08b5033acfb,68d081753ad4fe22fc4d410a9eb1ca01,delivered,2017-02-19 01:28:47,NaT,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27
39334,d77031d6a3c8a52f019764e68f211c69,0bf35cac6cc7327065da879e2d90fae8,delivered,2017-02-18 11:04:19,NaT,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22
48401,7002a78c79c519ac54022d4f8a65e6e8,d5de688c321096d15508faae67a27051,delivered,2017-01-19 22:26:59,NaT,2017-01-27 11:08:05,2017-02-06 14:22:19,2017-03-16
61743,2eecb0d85f281280f79fa00f9cec1a95,a3d3c38e58b9d2dfb9207cab690b6310,delivered,2017-02-17 17:21:55,NaT,2017-02-22 11:42:51,2017-03-03 12:16:03,2017-03-20


Данные заказы в масштабе всего датасета не имеют весомого влияния, поэтому их можно удалить из датафрейма

In [10]:
orders = orders[~orders['order_id'].isin(order_anomaly['order_id'])]

#### Датафрейм `products`

In [19]:
products[products['product_category_name'].isna()]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.00,17.00,14.00,12.00
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.00,16.00,7.00,20.00
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.00,20.00,20.00,20.00
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.00,41.00,30.00,41.00
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.00,35.00,7.00,12.00
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,NaN,NaN,NaN,1800.00,30.00,20.00,70.00
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,NaN,NaN,NaN,800.00,30.00,10.00,23.00
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,NaN,NaN,NaN,200.00,21.00,8.00,16.00
32772,fa51e914046aab32764c41356b9d4ea4,NaN,NaN,NaN,NaN,1300.00,45.00,16.00,45.00


### 1.3 Дубликаты

In [35]:
print(f'Дубликатов в датафрейме "customers": {customers.duplicated().sum()}')
print(f'Дубликатов в датафрейме "geo": {geo.duplicated().sum()}')
print(f'Дубликатов в датафрейме "order_items": {order_items.duplicated().sum()}')
print(f'Дубликатов в датафрейме "order_payments": {order_payments.duplicated().sum()}')
print(f'Дубликатов в датафрейме "order_reviews": {order_reviews.duplicated().sum()}')
print(f'Дубликатов в датафрейме "orders": {orders.duplicated().sum()}')
print(f'Дубликатов в датафрейме "products": {products.duplicated().sum()}')
print(f'Дубликатов в датафрейме "sellers": {sellers.duplicated().sum()}')
print(f'Дубликатов в датафрейме "product_category_name": {product_category_name.duplicated().sum()}')

Дубликатов в датафрейме "customers": 0
Дубликатов в датафрейме "geo": 261831
Дубликатов в датафрейме "order_items": 0
Дубликатов в датафрейме "order_payments": 0
Дубликатов в датафрейме "order_reviews": 0
Дубликатов в датафрейме "orders": 0
Дубликатов в датафрейме "products": 0
Дубликатов в датафрейме "sellers": 0
Дубликатов в датафрейме "product_category_name": 0


In [38]:
geo[geo.duplicated()].head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
15,1046,-23.55,-46.64,sao paulo,SP
44,1046,-23.55,-46.64,sao paulo,SP
65,1046,-23.55,-46.64,sao paulo,SP
66,1009,-23.55,-46.64,sao paulo,SP
67,1046,-23.55,-46.64,sao paulo,SP


Очень много дублирующихся строк геолокации. Удаляем дубликаты

In [40]:
geo = geo.drop_duplicates()
geo[geo.duplicated()].head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
